# Groups Detaied Data

In [1]:
"""
Ransomware Group Analysis Script
Analyzes 319 group JSON files from ransomware.live
Focus: tools, vulnerabilities, TF-IDF profiling
"""

import json
import os
import sys
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG - Cambia questo path con il tuo
# ============================================================
DATA_DIR = "/Users/dmk6603/Documents/ransom_victims/1-ransomware.live_data/data/groups_detailed"
OUTPUT_DIR = "output_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. LOAD ALL GROUP JSON FILES
# ============================================================
print("=" * 60)
print("1. LOADING GROUP DATA")
print("=" * 60)

groups = []
load_errors = []

for filename in sorted(os.listdir(DATA_DIR)):
    if not filename.endswith('.json'):
        continue
    filepath = os.path.join(DATA_DIR, filename)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            data['_filename'] = filename
            groups.append(data)
    except Exception as e:
        load_errors.append((filename, str(e)))

print(f"Loaded: {len(groups)} groups")
print(f"Errors: {len(load_errors)} files")
if load_errors:
    for fn, err in load_errors[:5]:
        print(f"  - {fn}: {err}")

# ============================================================
# 2. FIELD COMPLETENESS OVERVIEW
# ============================================================
print("\n" + "=" * 60)
print("2. FIELD COMPLETENESS OVERVIEW")
print("=" * 60)

fields_of_interest = [
    'group', 'description', 'victims', 'firstseen', 'lastseen',
    'ttps', 'vulnerabilities', 'tools', 'locations',
    'has_negotiations', 'negotiation_count',
    'has_ransomnote', 'ransomnotes_count'
]

completeness = {}
for field in fields_of_interest:
    count = 0
    for g in groups:
        val = g.get(field)
        if val is None:
            continue
        if isinstance(val, list) and len(val) == 0:
            continue
        if isinstance(val, dict) and len(val) == 0:
            continue
        if isinstance(val, str) and val.strip() == '':
            continue
        count += 1
    completeness[field] = count

completeness_df = pd.DataFrame([
    {'field': k, 'groups_with_data': v, 'pct': f"{v/len(groups)*100:.1f}%"}
    for k, v in completeness.items()
]).sort_values('groups_with_data', ascending=False)

print(completeness_df.to_string(index=False))
completeness_df.to_csv(os.path.join(OUTPUT_DIR, 'field_completeness.csv'), index=False)

# Bar chart
plt.figure(figsize=(10, 6))
plt.barh(completeness_df['field'], completeness_df['groups_with_data'], color='steelblue')
plt.xlabel('Number of Groups with Data')
plt.title('Field Completeness Across 319 Ransomware Groups')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'field_completeness.png'), dpi=150)
plt.close()
print("Saved: field_completeness.png")

# ============================================================
# 3. TOOLS ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("3. TOOLS ANALYSIS")
print("=" * 60)

# 3a. Collect all tools per group
groups_with_tools = []
tool_categories_all = set()
all_tools_flat = []  # (group, category, tool)

for g in groups:
    tools = g.get('tools')
    if not tools or not isinstance(tools, dict) or len(tools) == 0:
        continue
    group_name = g.get('group', g['_filename'].replace('.json', ''))
    groups_with_tools.append(group_name)
    for category, tool_list in tools.items():
        tool_categories_all.add(category)
        for tool in tool_list:
            all_tools_flat.append({
                'group': group_name,
                'category': category,
                'tool': tool.strip()
            })

tools_df = pd.DataFrame(all_tools_flat)
print(f"Groups with tools data: {len(groups_with_tools)}/{len(groups)}")
print(f"Total tool entries: {len(tools_df)}")
print(f"Unique tools: {tools_df['tool'].nunique()}")
print(f"Tool categories: {sorted(tool_categories_all)}")

# 3b. Most common tools overall
print("\n--- Top 20 Most Common Tools ---")
top_tools = tools_df['tool'].value_counts().head(20)
print(top_tools.to_string())

plt.figure(figsize=(12, 8))
top_tools.plot(kind='barh', color='coral')
plt.xlabel('Number of Groups Using This Tool')
plt.title('Top 20 Most Common Ransomware Tools')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'top_tools.png'), dpi=150)
plt.close()
print("Saved: top_tools.png")

# 3c. Tool category distribution
print("\n--- Tool Category Distribution ---")
cat_counts = tools_df.groupby('category')['group'].nunique().sort_values(ascending=False)
print(cat_counts.to_string())

plt.figure(figsize=(10, 6))
cat_counts.plot(kind='barh', color='mediumpurple')
plt.xlabel('Number of Groups')
plt.title('Tool Categories: How Many Groups Use Each Type')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'tool_categories.png'), dpi=150)
plt.close()
print("Saved: tool_categories.png")

# 3d. TF-IDF on tools (group = document, tools = terms)
print("\n--- TF-IDF: Tool Profiles per Group ---")

# Build document per group: all tool names concatenated
tool_docs = (
    tools_df
    .groupby('group')['tool']
    .apply(lambda x: ' '.join(x.str.lower().str.strip()))
    .reset_index()
)
tool_docs.columns = ['group', 'tool_doc']

# Only groups with at least 3 tools for meaningful TF-IDF
tool_count_per_group = tools_df.groupby('group')['tool'].count()
groups_enough_tools = tool_count_per_group[tool_count_per_group >= 3].index
tool_docs_filtered = tool_docs[tool_docs['group'].isin(groups_enough_tools)]

if len(tool_docs_filtered) >= 5:
    vec_tools = TfidfVectorizer(
        token_pattern=r'(?u)\b[\w\.\-]+\b',  # handle tool names with dots/hyphens
        min_df=1
    )
    tfidf_tools_matrix = vec_tools.fit_transform(tool_docs_filtered['tool_doc'])
    tool_features = vec_tools.get_feature_names_out()

    tfidf_tools_df = pd.DataFrame(
        tfidf_tools_matrix.toarray(),
        index=tool_docs_filtered['group'],
        columns=tool_features
    )

    # Top distinctive tools per group
    for group in tfidf_tools_df.index[:15]:  # first 15
        top = tfidf_tools_df.loc[group].nlargest(5)
        top = top[top > 0]
        if len(top) > 0:
            print(f"\n  {group}:")
            for tool, score in top.items():
                print(f"    {tool}: {score:.3f}")

    tfidf_tools_df.to_csv(os.path.join(OUTPUT_DIR, 'tfidf_tools.csv'))
    print("\nSaved: tfidf_tools.csv")

    # Cosine similarity between groups based on toolkits
    tool_sim = cosine_similarity(tfidf_tools_matrix)
    tool_sim_df = pd.DataFrame(
        tool_sim,
        index=tool_docs_filtered['group'],
        columns=tool_docs_filtered['group']
    )

    # Heatmap of top 25 groups by tool count
    top_tool_groups = tool_count_per_group.sort_values(ascending=False).head(25).index
    top_tool_groups = [g for g in top_tool_groups if g in tool_sim_df.index]

    if len(top_tool_groups) >= 5:
        plt.figure(figsize=(16, 14))
        sns.heatmap(
            tool_sim_df.loc[top_tool_groups, top_tool_groups],
            cmap='coolwarm',
            annot=True,
            fmt='.2f',
            linewidths=0.3,
            vmin=0, vmax=1
        )
        plt.title('Cosine Similarity Between Ransomware Groups (Based on Tools)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'tool_similarity_heatmap.png'), dpi=150)
        plt.close()
        print("Saved: tool_similarity_heatmap.png")

    tool_sim_df.to_csv(os.path.join(OUTPUT_DIR, 'tool_similarity.csv'))
else:
    print("Not enough groups with tools for TF-IDF analysis.")

# 3e. Tool co-occurrence matrix
print("\n--- Tool Co-occurrence (which tools appear together) ---")
group_tool_sets = tools_df.groupby('group')['tool'].apply(set).to_dict()
all_unique_tools = tools_df['tool'].unique()

# Limit to tools used by at least 3 groups
tool_group_count = tools_df.groupby('tool')['group'].nunique()
common_tools = tool_group_count[tool_group_count >= 3].index.tolist()

cooccurrence = pd.DataFrame(0, index=common_tools, columns=common_tools)
for group, toolset in group_tool_sets.items():
    relevant = [t for t in toolset if t in common_tools]
    for i, t1 in enumerate(relevant):
        for t2 in relevant[i:]:
            cooccurrence.loc[t1, t2] += 1
            if t1 != t2:
                cooccurrence.loc[t2, t1] += 1

if len(common_tools) > 3:
    # Show top 25 most common tools
    top_common = tool_group_count.sort_values(ascending=False).head(25).index
    top_common = [t for t in top_common if t in common_tools]

    plt.figure(figsize=(16, 14))
    sns.heatmap(
        cooccurrence.loc[top_common, top_common],
        cmap='YlOrRd',
        annot=True,
        fmt='d',
        linewidths=0.3
    )
    plt.title('Tool Co-occurrence Matrix (Top 25 Tools Used by 3+ Groups)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'tool_cooccurrence.png'), dpi=150)
    plt.close()
    print("Saved: tool_cooccurrence.png")

cooccurrence.to_csv(os.path.join(OUTPUT_DIR, 'tool_cooccurrence.csv'))

# ============================================================
# 4. VULNERABILITIES ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("4. VULNERABILITIES ANALYSIS")
print("=" * 60)

all_vulns = []
for g in groups:
    vulns = g.get('vulnerabilities')
    if not vulns or not isinstance(vulns, list) or len(vulns) == 0:
        continue
    group_name = g.get('group', g['_filename'].replace('.json', ''))
    for vuln in vulns:
        all_vulns.append({
            'group': group_name,
            'cve': vuln.strip() if isinstance(vuln, str) else str(vuln)
        })

vulns_df = pd.DataFrame(all_vulns)

if len(vulns_df) > 0:
    print(f"Groups with vulnerability data: {vulns_df['group'].nunique()}/{len(groups)}")
    print(f"Total CVE entries: {len(vulns_df)}")
    print(f"Unique CVEs: {vulns_df['cve'].nunique()}")

    # Most exploited CVEs
    print("\n--- Top 20 Most Exploited CVEs ---")
    top_cves = vulns_df['cve'].value_counts().head(20)
    print(top_cves.to_string())

    plt.figure(figsize=(12, 8))
    top_cves.plot(kind='barh', color='tomato')
    plt.xlabel('Number of Groups Exploiting This CVE')
    plt.title('Top 20 Most Exploited Vulnerabilities by Ransomware Groups')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'top_cves.png'), dpi=150)
    plt.close()
    print("Saved: top_cves.png")

    # CVEs per group
    cves_per_group = vulns_df.groupby('group')['cve'].count().sort_values(ascending=False)
    print("\n--- CVEs per Group (top 20) ---")
    print(cves_per_group.head(20).to_string())

    plt.figure(figsize=(12, 8))
    cves_per_group.head(25).plot(kind='barh', color='darkorange')
    plt.xlabel('Number of CVEs Exploited')
    plt.title('Ransomware Groups by Number of Known Exploited CVEs')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cves_per_group.png'), dpi=150)
    plt.close()
    print("Saved: cves_per_group.png")

    # CVE co-exploitation: which CVEs are exploited by the same groups?
    group_cve_sets = vulns_df.groupby('group')['cve'].apply(set).to_dict()
    cve_group_count = vulns_df.groupby('cve')['group'].nunique()
    shared_cves = cve_group_count[cve_group_count >= 2].index.tolist()

    if len(shared_cves) >= 3:
        cve_cooccurrence = pd.DataFrame(0, index=shared_cves, columns=shared_cves)
        for group, cveset in group_cve_sets.items():
            relevant = [c for c in cveset if c in shared_cves]
            for i, c1 in enumerate(relevant):
                for c2 in relevant[i:]:
                    cve_cooccurrence.loc[c1, c2] += 1
                    if c1 != c2:
                        cve_cooccurrence.loc[c2, c1] += 1

        # Show top 20
        top_shared = cve_group_count.loc[shared_cves].sort_values(ascending=False).head(20).index

        plt.figure(figsize=(16, 14))
        sns.heatmap(
            cve_cooccurrence.loc[top_shared, top_shared],
            cmap='Reds',
            annot=True,
            fmt='d',
            linewidths=0.3
        )
        plt.title('CVE Co-exploitation Matrix (Which CVEs Are Exploited by the Same Groups)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'cve_cooccurrence.png'), dpi=150)
        plt.close()
        print("Saved: cve_cooccurrence.png")

        cve_cooccurrence.to_csv(os.path.join(OUTPUT_DIR, 'cve_cooccurrence.csv'))

    # TF-IDF on vulnerabilities
    print("\n--- TF-IDF: Vulnerability Profiles per Group ---")
    vuln_docs = (
        vulns_df
        .groupby('group')['cve']
        .apply(lambda x: ' '.join(x.str.lower().str.replace('-', '_')))
        .reset_index()
    )
    vuln_docs.columns = ['group', 'vuln_doc']

    # Only groups with 2+ CVEs
    vuln_count_per_group = vulns_df.groupby('group')['cve'].count()
    groups_enough_vulns = vuln_count_per_group[vuln_count_per_group >= 2].index
    vuln_docs_filtered = vuln_docs[vuln_docs['group'].isin(groups_enough_vulns)]

    if len(vuln_docs_filtered) >= 5:
        vec_vulns = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
        tfidf_vulns_matrix = vec_vulns.fit_transform(vuln_docs_filtered['vuln_doc'])

        vuln_sim = cosine_similarity(tfidf_vulns_matrix)
        vuln_sim_df = pd.DataFrame(
            vuln_sim,
            index=vuln_docs_filtered['group'],
            columns=vuln_docs_filtered['group']
        )

        plt.figure(figsize=(16, 14))
        sns.heatmap(
            vuln_sim_df,
            cmap='coolwarm',
            annot=True if len(vuln_sim_df) <= 25 else False,
            fmt='.2f' if len(vuln_sim_df) <= 25 else '',
            linewidths=0.3,
            vmin=0, vmax=1
        )
        plt.title('Cosine Similarity Between Groups (Based on Exploited CVEs)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'vuln_similarity_heatmap.png'), dpi=150)
        plt.close()
        print("Saved: vuln_similarity_heatmap.png")

        vuln_sim_df.to_csv(os.path.join(OUTPUT_DIR, 'vuln_similarity.csv'))
else:
    print("No vulnerability data found in any group file.")

# ============================================================
# 5. COMBINED TOOLS + VULNERABILITIES PROFILE
# ============================================================
print("\n" + "=" * 60)
print("5. COMBINED ANALYSIS: TOOLS + CVEs")
print("=" * 60)

# Build combined document per group
combined_docs = {}
for g in groups:
    group_name = g.get('group', g['_filename'].replace('.json', ''))
    parts = []

    # Add tools
    tools = g.get('tools')
    if tools and isinstance(tools, dict):
        for cat, tool_list in tools.items():
            # Include category as a term too
            parts.append(cat.lower())
            for t in tool_list:
                parts.append(t.strip().lower())

    # Add vulnerabilities
    vulns = g.get('vulnerabilities')
    if vulns and isinstance(vulns, list):
        for v in vulns:
            parts.append(str(v).strip().lower().replace('-', '_'))

    if len(parts) >= 3:
        combined_docs[group_name] = ' '.join(parts)

print(f"Groups with combined tools+CVEs data (>=3 terms): {len(combined_docs)}")

if len(combined_docs) >= 5:
    combined_df = pd.DataFrame([
        {'group': k, 'doc': v} for k, v in combined_docs.items()
    ])

    vec_combined = TfidfVectorizer(
        token_pattern=r'(?u)\b[\w\.\-\_]+\b',
        min_df=1
    )
    tfidf_combined_matrix = vec_combined.fit_transform(combined_df['doc'])

    combined_sim = cosine_similarity(tfidf_combined_matrix)
    combined_sim_df = pd.DataFrame(
        combined_sim,
        index=combined_df['group'],
        columns=combined_df['group']
    )

    # Find most similar group pairs
    pairs = []
    for i in range(len(combined_sim_df)):
        for j in range(i + 1, len(combined_sim_df)):
            pairs.append({
                'group_1': combined_sim_df.index[i],
                'group_2': combined_sim_df.columns[j],
                'similarity': combined_sim[i][j]
            })

    pairs_df = pd.DataFrame(pairs).sort_values('similarity', ascending=False)
    print("\n--- Top 20 Most Similar Group Pairs (Tools + CVEs) ---")
    print(pairs_df.head(20).to_string(index=False))
    pairs_df.to_csv(os.path.join(OUTPUT_DIR, 'group_similarity_pairs.csv'), index=False)

    # Clustering
    if len(combined_sim_df) >= 10:
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0.7,
            metric='precomputed',
            linkage='average'
        )
        distance_matrix = 1 - combined_sim
        np.fill_diagonal(distance_matrix, 0)
        labels = clustering.fit_predict(distance_matrix)

        cluster_df = pd.DataFrame({
            'group': combined_df['group'],
            'cluster': labels
        }).sort_values('cluster')

        print(f"\n--- Clusters Found: {cluster_df['cluster'].nunique()} ---")
        for c in sorted(cluster_df['cluster'].unique()):
            members = cluster_df[cluster_df['cluster'] == c]['group'].tolist()
            if len(members) > 1:
                print(f"\n  Cluster {c} ({len(members)} groups):")
                for m in members:
                    print(f"    - {m}")

        cluster_df.to_csv(os.path.join(OUTPUT_DIR, 'group_clusters.csv'), index=False)
        print("Saved: group_clusters.csv")

    # Heatmap for top 30 groups
    top_combined_groups = (
        combined_df['doc'].str.split().apply(len)
        .sort_values(ascending=False)
        .head(30)
        .index
    )
    top_combined_names = combined_df.loc[top_combined_groups, 'group'].tolist()
    top_combined_names = [n for n in top_combined_names if n in combined_sim_df.index]

    plt.figure(figsize=(18, 16))
    sns.heatmap(
        combined_sim_df.loc[top_combined_names, top_combined_names],
        cmap='coolwarm',
        annot=True if len(top_combined_names) <= 20 else False,
        fmt='.2f' if len(top_combined_names) <= 20 else '',
        linewidths=0.2,
        vmin=0, vmax=1
    )
    plt.title('Group Similarity (Combined Tools + CVEs Profile)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'combined_similarity_heatmap.png'), dpi=150)
    plt.close()
    print("Saved: combined_similarity_heatmap.png")

# ============================================================
# 6. SUMMARY STATS TABLE
# ============================================================
print("\n" + "=" * 60)
print("6. GROUP SUMMARY TABLE")
print("=" * 60)

summary_rows = []
for g in groups:
    group_name = g.get('group', g['_filename'].replace('.json', ''))

    n_tools = 0
    n_tool_categories = 0
    tools = g.get('tools')
    if tools and isinstance(tools, dict):
        n_tool_categories = len(tools)
        n_tools = sum(len(v) for v in tools.values())

    n_vulns = 0
    vulns = g.get('vulnerabilities')
    if vulns and isinstance(vulns, list):
        n_vulns = len(vulns)

    n_ttps = 0
    ttps = g.get('ttps')
    if ttps and isinstance(ttps, list):
        n_ttps = len(ttps)

    n_locations = 0
    locs = g.get('locations')
    if locs and isinstance(locs, list):
        n_locations = len(locs)

    summary_rows.append({
        'group': group_name,
        'victims': g.get('victims', 0) or 0,
        'firstseen': g.get('firstseen', ''),
        'lastseen': g.get('lastseen', ''),
        'n_tools': n_tools,
        'n_tool_categories': n_tool_categories,
        'n_vulnerabilities': n_vulns,
        'n_ttps': n_ttps,
        'has_negotiations': g.get('has_negotiations', False),
        'negotiation_count': g.get('negotiation_count', 0) or 0,
        'has_ransomnote': g.get('has_ransomnote', False),
        'ransomnotes_count': g.get('ransomnotes_count', 0) or 0,
        'n_onion_sites': n_locations,
        'has_description': bool(g.get('description', ''))
    })

summary_df = pd.DataFrame(summary_rows).sort_values('victims', ascending=False)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'group_summary.csv'), index=False)

print(f"Total groups: {len(summary_df)}")
print(f"Groups with tools: {(summary_df['n_tools'] > 0).sum()}")
print(f"Groups with CVEs: {(summary_df['n_vulnerabilities'] > 0).sum()}")
print(f"Groups with TTPs: {(summary_df['n_ttps'] > 0).sum()}")
print(f"Groups with negotiations: {(summary_df['has_negotiations']).sum()}")
print(f"Groups with ransom notes: {(summary_df['has_ransomnote']).sum()}")

print(f"\n--- Top 20 Groups by Victim Count ---")
print(summary_df[['group', 'victims', 'n_tools', 'n_vulnerabilities']].head(20).to_string(index=False))

# Scatter: victims vs tools
plt.figure(figsize=(10, 8))
scatter_df = summary_df[summary_df['n_tools'] > 0]
plt.scatter(scatter_df['n_tools'], scatter_df['victims'], alpha=0.6, s=50, color='steelblue')
for _, row in scatter_df.iterrows():
    if row['victims'] > 500 or row['n_tools'] > 20:
        plt.annotate(row['group'], (row['n_tools'], row['victims']),
                     fontsize=8, alpha=0.8)
plt.xlabel('Number of Known Tools')
plt.ylabel('Number of Victims')
plt.title('Ransomware Groups: Victims vs Known Tool Arsenal')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'victims_vs_tools.png'), dpi=150)
plt.close()
print("\nSaved: victims_vs_tools.png")

# ============================================================
print("\n" + "=" * 60)
print("DONE! All outputs saved to:", OUTPUT_DIR)
print("=" * 60)
print("\nFiles generated:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

1. LOADING GROUP DATA
Loaded: 319 groups
Errors: 0 files

2. FIELD COMPLETENESS OVERVIEW
            field  groups_with_data    pct
            group               319 100.0%
          victims               319 100.0%
 has_negotiations               319 100.0%
negotiation_count               319 100.0%
   has_ransomnote               319 100.0%
ransomnotes_count               319 100.0%
        locations               317  99.4%
        firstseen               251  78.7%
         lastseen               251  78.7%
      description                99  31.0%
            tools                60  18.8%
             ttps                21   6.6%
  vulnerabilities                 6   1.9%
Saved: field_completeness.png

3. TOOLS ANALYSIS
Groups with tools data: 60/319
Total tool entries: 689
Unique tools: 203
Tool categories: ['CredentialTheft', 'DefenseEvasion', 'DiscoveryEnum', 'Exfiltration', 'LOLBAS', 'Networking', 'Offsec', 'RMM-Tools']

--- Top 20 Most Common Tools ---
tool
Cobalt Strike